In [ ]:
# Colab setup: install pyedautils; elsewhere load plotly.js from a CDN for the plots
import sys
if "google.colab" in sys.modules:
    %pip install -q "pyedautils>=0.0.32"
else:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"

# Hourly Duration Curve

```{admonition} Goal
:class: tip
Display a duration curve (sum frequency) of hourly electricity consumption. The curve ranks hourly totals from highest to lowest, showing the distribution of consumption intensities.
```

```{admonition} Data Basis
:class: note
The dataset `flatElectricity.csv` contains cumulative electricity meter readings at 15-minute intervals (columns: `time`, `FlatA_Ele`, `FlatB_Ele`, `FlatC_Ele`, `FlatD_Ele`; separator `;`). We use `FlatA_Ele`, compute consumption via `diff()`, and sum it per hour, keeping only hours without missing readings.
```

## Data Preparation

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatElectricity.csv"
raw = pd.read_csv(url, sep=";")

# Parse timestamp and set as index
raw["time"] = pd.to_datetime(raw["time"])
raw = raw.set_index("time")

# Consumption per 15 minutes from the cumulative meter readings (kWh)
consumption = raw["FlatA_Ele"].diff()

# Hourly sums of complete hours only (all 4 quarter-hours have a reading)
hourly = consumption.resample("h").sum(min_count=4).dropna()

# Build clean 2-column DataFrame [timestamp, value]
df = hourly.reset_index()
df.columns = ["timestamp", "value"]
df.head()

## Visualization

In [ ]:
from pyedautils.plots import plot_sum_frequency

fig = plot_sum_frequency(df, resolution="hourly", reverse=True,
                         title="Electricity — Hourly Duration Curve", ylab="Electricity [kWh/h]")
fig.show()

::::{dropdown} Customization
`reverse=True` puts the highest value first (classic duration curve); without it the values are sorted from lowest to highest. `resolution` is `"hourly"` or `"daily"`, and `year=2019` restricts the curve to one year.

```python
fig = plot_sum_frequency(df, resolution="hourly", reverse=True, year=2019, ylab="Electricity [kWh/h]")
```
::::